# 7BDNI013C — Data Mining and Machine Learning
## Mortgage Loan Application Outcome Prediction
**Dataset:** HMDA Public Loan-Level Dataset — Pennsylvania, Actions 1-2-3, Loan Purpose 1  
**Source:** Consumer Financial Protection Bureau (CFPB)  
**Target:** `action_taken` — 0 = Application Denied, 1 = Approved but Not Accepted, 2 = Loan Originated



## 0. Imports & Global Config


In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42

## 1. Data Loading

The dataset is loaded from the raw HMDA CSV file. A copy is made immediately so the original is never modified.  
`low_memory=False` is required because several columns contain mixed types (numeric codes and exempt markers) that pandas cannot infer from the first rows alone.


In [ ]:
raw = pd.read_csv('../actions_taken_1-2-3_loan_purposes_1_state_PA.csv', low_memory=False)
copy = raw.copy()
print(f"Dataset shape: {copy.shape[0]:,} rows × {copy.shape[1]} columns")

## 2. Initial Data Audit

`auditor()` inspects every column for data type, missing value count, and unique value count.  
This gives a structural overview of the raw dataset before any cleaning is applied.


In [ ]:
import pandas as pd

def auditor(df: pd.DataFrame) -> pd.DataFrame:
    audit_data = []

    for col in df.columns:
        dtype = df[col].dtypes
        missing_count = df[col].isna().sum()
        unique_count = df[col].nunique()
        audit_data.append({
            'column': col,
            'dtype': str(dtype),
            'missing_count': missing_count,
            'unique_val': unique_count
        })
    audit_df = pd.DataFrame(audit_data)
    return audit_df

audit_raw = auditor(copy)
audit_raw

## 3. Data Cleaning

Cleaning is applied in four sequential steps:

1. **`one_value()`** — drops columns with only a single unique value (zero information content)  
2. **`cardinals()`** — drops `lei`, a unique legal entity identifier per institution (effectively a row ID, cannot generalise)  
3. **`leaker()`** — drops columns that would cause data leakage: `purchaser_type` (known only after loan sale) and `denial_reason-*` (directly encodes the target outcome)  
4. **`high_missing()`** — drops columns where ≥50% of values are missing (imputing beyond this threshold manufactures more data than it recovers)  
5. **`transformer()`** — converts columns stored as strings to their correct numeric types (HMDA stores exempt/NA codes mixed into numeric fields, causing string dtype)


In [ ]:
import pandas as pd

shape_before = copy.shape


def one_value(df: pd.DataFrame) -> pd.DataFrame:
    """Drop columns that have only a single unique value — they carry no information."""
    uniq_1 = df.columns[df.nunique() == 1]
    return df.drop(columns=uniq_1)


def cardinals(df: pd.DataFrame) -> pd.DataFrame:
    """Drop extremely high-cardinality identifiers that cannot be generalised.
    'lei' is a unique legal entity identifier per institution — effectively an ID column."""
    cardinal = ["lei"]
    return df.drop(columns=cardinal)


def leaker(df: pd.DataFrame) -> pd.DataFrame:
    """Drop columns that would cause data leakage.
    purchaser_type is only known after a loan is sold (post-decision).
    denial_reason-* directly encode the target outcome and would leak the label."""
    leakers = ["purchaser_type", "denial_reason-1", "denial_reason-2",
               "denial_reason-3", "denial_reason-4"]
    return df.drop(columns=leakers, errors="ignore")


def high_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Drop columns where more than 50% of values are missing.
    At that level of missingness, imputation would be manufacturing data
    rather than recovering it, introducing more noise than signal."""
    missers = [col for col in df.columns
               if (df[col].isna().sum() / df.shape[0]) * 100 >= 50]
    return df.drop(columns=missers, errors="ignore")


def transformer(df: pd.DataFrame) -> pd.DataFrame:
    """Convert columns stored as strings to their correct numeric types.
    These arrive as strings in the raw HMDA file due to exempt/NA codes
    mixed into otherwise numeric fields."""
    temp_df = df.copy()

    numeric_cols = [
        'loan_to_value_ratio', 'interest_rate', 'rate_spread',
        'total_loan_costs', 'origination_charges', 'loan_term',
        'property_value', 'debt_to_income_ratio', 'total_units'
    ]

    for col in numeric_cols:
        if col in temp_df.columns:
            temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce')

    return temp_df

copy = one_value(copy)
copy = cardinals(copy)
copy = leaker(copy)
copy = high_missing(copy)
copy = transformer(copy)
shape_after = copy.shape

print(f"Before cleaning: {shape_before[0]:,} rows × {shape_before[1]} columns")
print(f"After cleaning:  {shape_after[0]:,} rows × {shape_after[1]} columns")
print(f"Columns removed: {shape_before[1] - shape_after[1]}")

## 4. Feature Engineering & Mapping

`mapper()` transforms raw HMDA integer codes into human-readable string labels and engineers new features:

- All integer-coded columns (loan type, race, sex, AUS system, etc.) are mapped to their HMDA codebook descriptions. This ensures the model receives meaningful categorical signals rather than arbitrary integers.
- `total_units` is converted to a binary `is_single_family` flag — the most important distinction for mortgage risk is single-family vs. multi-unit, not the exact unit count.
- `applicant_age` and `co-applicant_age` are ordinally encoded (1–7 for age bands). Code `8888` (Not Applicable) is captured as a separate binary flag. Code `9999` (No Co-applicant) is captured as `no_coapplicant` binary flag — preserving information that would otherwise be lost.
- `hoepa_status` was mapped in an earlier version but was removed because it was not present in the model's feature list and was silently dropped by the preprocessor.

**Note:** `action_taken` is mapped here for EDA readability. It is re-encoded numerically (0/1/2) in the modelling section.


In [ ]:
import pandas as pd
def mapper(df: pd.DataFrame) -> pd.DataFrame:
    temp_df = df.copy()

    # conforming_loan_limit
    temp_df['conforming_loan_limit'] = temp_df['conforming_loan_limit'].map({
        'C': 'Conforming',
        'NC': 'Non-conforming',
        'U': 'Undetermined'})
    temp_df['conforming_loan_limit'] = (temp_df['conforming_loan_limit'].fillna('Missing')
                                        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # derived_dwelling_category
    temp_df['derived_dwelling_category'] = temp_df['derived_dwelling_category'].map({
        "Single Family (1-4 Units):Site-Built": 1,
        "Single Family (1-4 Units):Manufactured": 2,
        "Multifamily:Site-Built": 3,
        "Multifamily:Manufactured": 4})
    temp_df['derived_dwelling_category'] = temp_df['derived_dwelling_category'].fillna(0).astype(int)


    # action_taken
    temp_df['action_taken'] = temp_df['action_taken'].map({
        1: 'Loan originated',
        2: 'Approved but not accepted',
        3: 'Application denied'})
    temp_df['action_taken'] = (temp_df['action_taken'].fillna('Missing')
                               .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # preapproval
    temp_df['preapproval'] = temp_df['preapproval'].map({
        1: 'Preapproval requested',
        2: 'Not requested'})
    temp_df['preapproval'] = (temp_df['preapproval'].fillna('Missing')
                              .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # loan_type
    temp_df['loan_type'] = temp_df['loan_type'].map({
        1: 'Conventional',
        2: 'FHA',
        3: 'VA',
        4: 'USDA RHS/FSA'})
    temp_df['loan_type'] = (temp_df['loan_type'].fillna('Missing')
                            .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # lien_status
    temp_df['lien_status'] = temp_df['lien_status'].map({
        1: 'First lien',
        2: 'Subordinate'})
    temp_df['lien_status'] = (temp_df['lien_status'].fillna('Missing')
                              .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # reverse_mortgage
    temp_df['reverse_mortgage'] = temp_df['reverse_mortgage'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt' })
    temp_df['reverse_mortgage'] = (temp_df['reverse_mortgage'].fillna('Missing')
                                   .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # open-end_line_of_credit
    temp_df['open-end_line_of_credit'] = temp_df['open-end_line_of_credit'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt' })
    temp_df['open-end_line_of_credit'] = (temp_df['open-end_line_of_credit'].fillna('Missing')
                                          .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # business_or_commercial_purpose
    temp_df['business_or_commercial_purpose'] = temp_df['business_or_commercial_purpose'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt'})
    temp_df['business_or_commercial_purpose'] = (temp_df['business_or_commercial_purpose'].fillna('Missing')
                                                 .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # negative_amortization
    temp_df['negative_amortization'] = temp_df['negative_amortization'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt'})
    temp_df['negative_amortization'] = (temp_df['negative_amortization'].fillna('Missing')
                                        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # interest_only_payment
    temp_df['interest_only_payment'] = temp_df['interest_only_payment'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt'})
    temp_df['interest_only_payment'] = (temp_df['interest_only_payment'].fillna('Missing')
                                        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # balloon_payment
    temp_df['balloon_payment'] = temp_df['balloon_payment'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt' })
    temp_df['balloon_payment'] = (temp_df['balloon_payment'].fillna('Missing')
                                  .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # other_nonamortizing_features
    temp_df['other_nonamortizing_features'] = temp_df['other_nonamortizing_features'].map({
        1: 'Yes',
        2: 'No',
        1111: 'Exempt'})
    temp_df['other_nonamortizing_features'] = (temp_df['other_nonamortizing_features'].fillna('Missing')
                                               .astype('string').str.strip().str.replace(' ', '_', regex=False))

    # construction_method
    temp_df['construction_method'] = temp_df['construction_method'].map({
        1: 'Site-built',
        2: 'Manufactured home'})
    temp_df['construction_method'] = (temp_df['construction_method'].fillna('Missing')
                                      .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # occupancy_type
    temp_df['occupancy_type'] = temp_df['occupancy_type'].map({
        1: 'Principal residence',
        2: 'Second residence',
        3: 'Investment property'})
    temp_df['occupancy_type'] = (temp_df['occupancy_type'].fillna('Missing')
                                 .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # manufactured_home_secured_property_type
    temp_df['manufactured_home_secured_property_type'] = temp_df['manufactured_home_secured_property_type'].map({
        1: 'Home and land',
        2: 'Home not land',
        3: 'Not applicable',
        1111: 'Exempt'})
    temp_df['manufactured_home_secured_property_type'] = (
        temp_df['manufactured_home_secured_property_type'].fillna('Missing')
        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # manufactured_home_land_property_interest
    temp_df['manufactured_home_land_property_interest'] = (
        temp_df['manufactured_home_land_property_interest'].map({
        1: 'Direct ownership',
        2: 'Indirect ownership',
        3: 'Paid leasehold',
        4: 'Unpaid leasehold',
        5: 'Not applicable',
        1111: 'Exempt'}))
    temp_df['manufactured_home_land_property_interest'] = (
        temp_df['manufactured_home_land_property_interest'].fillna('Missing')
        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # total_units
    temp_df["is_single_family"] = (temp_df["total_units"] == 1).astype(int)
    temp_df = temp_df.drop(columns=["total_units"])


    # applicant_credit_score_type
    temp_df['applicant_credit_score_type'] = temp_df['applicant_credit_score_type'].map({
        1: 'Equifax Beacon 5.0',
        2: 'Experian Fair Isaac',
        3: 'FICO Classic 04',
        4: 'FICO Classic 98',
        5: 'VantageScore 2.0',
        6: 'VantageScore 3.0',
        7: 'More than one model',
        8: 'Other model',
        9: 'Not applicable',
        11: 'FICO Score 9',
        12: 'FICO Score 8',
        13: 'FICO Score 10',
        14: 'FICO Score 10T',
        15: 'VantageScore 4.0',
        1111: 'Exempt'})
    temp_df['applicant_credit_score_type'] = (
        temp_df['applicant_credit_score_type'].fillna('Missing')
        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # co-applicant_credit_score_type
    temp_df['co-applicant_credit_score_type'] = temp_df['co-applicant_credit_score_type'].map({
        1: 'Equifax Beacon 5.0',
        2: 'Experian Fair Isaac',
        3: 'FICO Classic 04',
        4: 'FICO Classic 98',
        5: 'VantageScore 2.0',
        6: 'VantageScore 3.0',
        7: 'More than one model',
        8: 'Other model',
        9: 'Not applicable',
        10: 'No co-applicant',
        11:'FICO Score 9',
        12: 'FICO Score 8',
        13: 'FICO Score 10',
        14: 'FICO Score 10T',
        15: 'VantageScore 4.0',
        1111: 'Exempt'})
    temp_df['co-applicant_credit_score_type'] = (
        temp_df['co-applicant_credit_score_type'].fillna('Missing')
        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # applicant_ethnicity - 1
    temp_df['applicant_ethnicity-1'] = temp_df['applicant_ethnicity-1'].map({
        1.0: 'Hispanic or Latino',
        2.0: 'Not Hispanic or Latino',
        3.0: 'Not provided',
        4.0: 'Not applicable',
        11.0: 'Mexican',
        12.0: 'Puerto Rican',
        13.0: 'Cuban',
        14.0: 'Other Hispanic or Latino'})
    temp_df['applicant_ethnicity-1'] = (
        temp_df['applicant_ethnicity-1'].fillna('Missing')
        .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # co-applicant_ethnicity-1
    temp_df['co-applicant_ethnicity-1'] = temp_df['co-applicant_ethnicity-1'].map({
        1.0: 'Hispanic or Latino',
        2.0: 'Not Hispanic or Latino',
        3.0: 'Not provided',
        4.0: 'Not applicable',
        5.0: 'No co-applicant',
        11.0: 'Mexican',
        12.0: 'Puerto Rican',
        13.0: 'Cuban',
        14.0: 'Other Hispanic or Latino'})
    temp_df['co-applicant_ethnicity-1'] = (temp_df['co-applicant_ethnicity-1'].fillna('Missing')
                                           .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # applicant_ethnicity_observed
    temp_df['applicant_ethnicity_observed'] = temp_df['applicant_ethnicity_observed'].map({
        1: 'Collected on visual observation or surname',
        2: 'Not collected on visual observation or surname',
        3: 'Not applicable',
        1111: 'Exempt'})
    temp_df['applicant_ethnicity_observed'] = (temp_df['applicant_ethnicity_observed'].fillna('Missing')
                                               .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # co-applicant_ethnicity_observed
    temp_df['co-applicant_ethnicity_observed'] = temp_df['co-applicant_ethnicity_observed'].map({
        1: 'Collected on visual observation or surname',
        2: 'Not collected on visual observation or surname',
        3: 'Not applicable',
        4: 'No co-applicant',
        1111: 'Exempt'})
    temp_df['co-applicant_ethnicity_observed'] = (temp_df['co-applicant_ethnicity_observed'].fillna('Missing')
                                                  .astype('string').str.strip().str.replace(' ', '_', regex=False))

    # applicant_race-1
    temp_df['applicant_race-1'] = pd.to_numeric(temp_df['applicant_race-1'], errors='coerce')
    temp_df['applicant_race-1'] = temp_df['applicant_race-1'].map({
        1.0: 'American Indian or Alaska Native',
        2.0: 'Asian',
        2.1: 'Asian Indian',
        2.2: 'Chinese',
        2.3: 'Filipino',
        2.4: 'Japanese',
        2.5: 'Korean',
        2.6: 'Vietnamese',
        2.7: 'Other Asian',
        3.0: 'Black or African American',
        4.0: 'Native Hawaiian or Other Pacific Islander',
        4.1: 'Native Hawaiian',
        4.2: 'Guamanian or Chamorro',
        4.3: 'Samoan',
        4.4: 'Other Pacific Islander',
        5.0: 'White',
        6.0: 'Information not provided by applicant in mail, internet, or telephone application',
        7.0: 'Not applicable'})
    temp_df['applicant_race-1'] = (temp_df['applicant_race-1'].fillna('Missing')
                                   .astype('string').str.strip().str.replace(' ', '_', regex=False))

    # co-applicant_race-1
    temp_df['co-applicant_race-1'] = pd.to_numeric(temp_df['co-applicant_race-1'], errors='coerce')
    temp_df['co-applicant_race-1'] = temp_df['co-applicant_race-1'].map({
        1.0: 'American Indian or Alaska Native',
        2.0: 'Asian',
        2.1: 'Asian Indian',
        2.2: 'Chinese',
        2.3: 'Filipino',
        2.4: 'Japanese',
        2.5: 'Korean',
        2.6: 'Vietnamese',
        2.7: 'Other Asian',
        3.0: 'Black or African American',4.0: 'Native Hawaiian or Other Pacific Islander',
        4.1: 'Native Hawaiian',
        4.2: 'Guamanian or Chamorro',
        4.3: 'Samoan',
        4.4: 'Other Pacific Islander',
        5.0: 'White',
        6.0: 'Information not provided by applicant in mail, internet, or telephone application',
        7.0: 'Not applicable',
        8.0: 'No co-applicant'})
    temp_df['co-applicant_race-1'] = (temp_df['co-applicant_race-1'].fillna('Missing')
                                      .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # applicant_race_observed
    temp_df['applicant_race_observed'] = temp_df['applicant_race_observed'].map({
        1: 'Collected on visual observation or surname',
        2: 'Not collected on visual observation or surname',
        3: 'Not applicable',
        1111: 'Exempt'})
    temp_df['applicant_race_observed'] = (temp_df['applicant_race_observed'].fillna('Missing')
                                          .astype('string').str.strip().str.replace(' ', '_', regex=False))

    # co-applicant_race_observed
    temp_df['co-applicant_race_observed'] = temp_df['co-applicant_race_observed'].map({
        1: 'Collected on visual observation or surname',
        2: 'Not collected on visual observation or surname',
        3: 'Not applicable',
        4: 'No co-applicant',
        1111: 'Exempt'})
    temp_df['co-applicant_race_observed'] = (temp_df['co-applicant_race_observed'].fillna('Missing')
                                             .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # applicant_sex
    temp_df['applicant_sex'] = temp_df['applicant_sex'].map({
        1: 'Male',
        2: 'Female',
        3: 'Information not provided by applicant in mail, internet, or telephone application',
        4: 'Not applicable',
        6: 'Applicant selected both male and female'})
    temp_df['applicant_sex'] = (temp_df['applicant_sex'].fillna('Missing')
                                .astype('string').str.strip().str.replace(' ', '_', regex=False))

    # co-applicant_sex
    temp_df['co-applicant_sex'] = temp_df['co-applicant_sex'].map({
        1: 'Male',
        2: 'Female',
        3: 'Information not provided by applicant in mail, internet, or telephone application',
        4: 'Not applicable',
        5: 'No co-applicant',
        6: 'Co-applicant selected both male and female'})
    temp_df['co-applicant_sex'] = (temp_df['co-applicant_sex'].fillna('Missing')
                                   .astype('string').str.strip().str.replace(' ', '_', regex=False))

    # applicant_sex_observed
    temp_df['applicant_sex_observed'] = temp_df['applicant_sex_observed'].map({
        1: 'Collected on visual observation or surname',
        2: 'Not collected on visual observation or surname',
        3: 'Not applicable',
        1111: 'Exempt'})
    temp_df['applicant_sex_observed'] = (temp_df['applicant_sex_observed'].fillna('Missing')
                                         .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # co-applicant_sex_observed
    temp_df['co-applicant_sex_observed'] = temp_df['co-applicant_sex_observed'].map({
        1: 'Collected on visual observation or surname',
        2: 'Not collected on visual observation or surname',
        3: 'Not applicable',
        4: 'No co-applicant',
        1111: 'Exempt'})
    temp_df['co-applicant_sex_observed'] = (temp_df['co-applicant_sex_observed'].fillna('Missing')
                                            .astype( 'string').str.strip().str.replace(' ', '_', regex=False))

    age_map = {
        '<25': 1,
        '25-34': 2,
        '35-44': 3,
        '45-54': 4,
        '55-64': 5,
        '65-74': 6,
        '>74': 7
    }

    # Applicant age
    temp_df['applicant_age_ord'] = (temp_df['applicant_age'].map(age_map).astype('Int8'))
    temp_df['applicant_age_not_applicable'] = (temp_df['applicant_age'].eq('8888').astype('int8'))
    temp_df['applicant_age_ord'] = (temp_df['applicant_age_ord'].fillna(0).astype('int8'))
    temp_df =temp_df.drop(columns=['applicant_age'])

    # Co-applicant age
    temp_df['coapplicant_age_ord'] = (temp_df['co-applicant_age'].map(age_map).astype('Int8'))
    temp_df['coapplicant_age_not_applicable'] = (temp_df['co-applicant_age'].eq('8888').astype('int8'))
    temp_df['no_coapplicant'] = (temp_df['co-applicant_age'].eq('9999').astype('int8'))
    temp_df['coapplicant_age_ord'] = (temp_df['coapplicant_age_ord'].fillna(0).astype('int8'))

    temp_df = temp_df.drop(columns=['co-applicant_age'])



    # applicant_age_above_62
    temp_df['applicant_age_above_62'] = (temp_df['applicant_age_above_62'].fillna('Exempt')
                                         .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # submission_of_application
    temp_df['submission_of_application'] = temp_df['submission_of_application'].map({
        1: 'Submitted directly to institution',
        2: 'Not submitted directly to institution',
        1111: 'Exempt'})
    temp_df['submission_of_application'] = (temp_df['submission_of_application'].fillna('Missing')
                                            .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # initially_payable_to_institution
    temp_df['initially_payable_to_institution'] = temp_df['initially_payable_to_institution'].map({
        1: 'Initially payable to your institution',
        2: 'Not initially payable to your institution',
        3: 'Not applicable',
        1111: 'Exempt'})
    temp_df['initially_payable_to_institution'] = (temp_df['initially_payable_to_institution'].fillna('Missing')
                                                   .astype('string').str.strip().str.replace(' ', '_', regex=False))


    # aus-1
    temp_df['aus-1'] = temp_df['aus-1'].map({
        1: 'Desktop Underwriter (DU)',
        2: 'Loan Product Advisor (LPA)',
        3: 'GUS',
        4: 'Other',
        5: 'Not applicable',
        6: 'Internal Proprietary AUS',
        1111: 'Exempt'})
    temp_df['aus-1'] = (temp_df['aus-1'].fillna('Missing')
                        .astype('string').str.strip().str.replace(' ', '_', regex=False))

    return temp_df

copy = mapper(copy)
print(f"Shape after mapping: {copy.shape[0]:,} rows × {copy.shape[1]} columns")
copy.head(3)

## 5. Post-Cleaning Summary

`summarizer()` provides a final structural overview after all cleaning and mapping steps.  
This confirms that data types are correct, missing values are at acceptable levels, and no unexpected columns remain.


In [ ]:
import pandas as pd

def summarizer(df: pd.DataFrame) -> pd.DataFrame:
    temp_df = df.copy()

    summary_data = {
        'Data Type': temp_df.dtypes,
        'Non-Null Count': temp_df.notnull().sum(),
        'Missing Count': temp_df.isnull().sum(),
        'Unique Values': temp_df.nunique(),
        'Sample Values': [
            temp_df[col].dropna().unique()[:1].tolist()
            for col in temp_df.columns]}

    summary_df = pd.DataFrame(summary_data)

    return summary_df

summary = summarizer(copy)
summary

## 6. Exploratory Data Analysis

The EDA focuses on understanding the target distribution, key numeric feature relationships with outcomes, and potential fairness signals in the data. Each visualisation is chosen to inform a specific preprocessing or modelling decision — not for completeness.

**Key findings to note:**
- The dataset is severely imbalanced (~86% Loan Originated). This directly motivates the choice of macro F1 as the evaluation metric and the use of `class_weight` in tree-based models.
- Denied applications cluster at higher DTI ratios and lower incomes — the model must learn these boundaries.
- Denial rates vary by race and neighbourhood minority population percentage — a fairness consideration noted in the business recommendation.
- Interest rate and rate spread show strong separation by outcome — these are post-decision leakers and have been removed before modelling.


In [ ]:
TARGET = 'action_taken'
ORDER  = ['Loan_originated', 'Approved_but_not_accepted', 'Application_denied']
COLORS = ['#2ecc71', '#f39c12', '#e74c3c']

### 6.1 Target Class Distribution

In [ ]:
def plot_target_distribution(df):
    counts = df[TARGET].value_counts()
    values = [counts[label] for label in ORDER]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(ORDER, values, color=COLORS)

    for bar, val in zip(bars, values):
        pct = val / len(df) * 100
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 200,
                f'{val:,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=9)

    ax.set_title('Action Taken - Class Distribution')
    ax.set_ylabel('Count')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Originated', 'Approved Not Accepted', 'Denied'])
    plt.tight_layout()
    plt.show()

plot_target_distribution(copy)

### 6.2 Numeric Features by Outcome

Income and loan amount are clipped to realistic ranges (0–500k and 0–1M respectively) to remove data entry errors and extreme outliers that would compress the box plots. The DTI panel uses a bar chart because DTI in this dataset is a bucketed ordinal variable, not a true continuous one.


In [ ]:
def plot_boxplots_numeric(df):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Income — clip between 0 and 500 (realistic income range in $k)
    ax = axes[0][0]
    plot_df = df[(df['income'] >= 0) & (df['income'] <= 500)]
    sns.boxplot(data=plot_df, x=TARGET, y='income',
                order=ORDER, hue=TARGET, palette=COLORS,
                legend=False, ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_title('Income (clipped 0–500k)')
    ax.set_xlabel('')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Originated', 'Approved\nNot Accepted', 'Denied'], fontsize=8)

    ax = axes[0][1]
    plot_df = df[df['loan_amount'] <= 1_000_000]
    sns.boxplot(data=plot_df, x=TARGET, y='loan_amount',
                order=ORDER, hue=TARGET, palette=COLORS,
                legend=False, ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_title('Loan Amount (clipped at $1M)')
    ax.set_xlabel('')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Originated', 'Approved\nNot Accepted', 'Denied'], fontsize=8)

    ax = axes[1][0]
    sns.boxplot(data=df, x=TARGET, y='interest_rate',
                order=ORDER, hue=TARGET, palette=COLORS,
                legend=False, ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_title('Interest Rate')
    ax.set_xlabel('')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Originated', 'Approved\nNot Accepted', 'Denied'], fontsize=8)

    ax = axes[1][1]
    dti_counts = df.groupby([TARGET, 'debt_to_income_ratio']).size().reset_index(name='count')
    for outcome, color in zip(ORDER, COLORS):
        subset = dti_counts[dti_counts[TARGET] == outcome]
        ax.bar(subset['debt_to_income_ratio'], subset['count'],
               alpha=0.6, color=color, label=outcome.replace('_', ' '), width=0.4)
    ax.set_title('Debt-to-Income Ratio Distribution')
    ax.set_xlabel('DTI Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

    plt.tight_layout()
    plt.show()

plot_boxplots_numeric(copy)

### 6.3 Loan-to-Value Ratio Distribution

Values above 105% are excluded as dirty data (LTV cannot physically exceed 100% for standard loans). The 80% and 100% reference lines mark the conventional PMI threshold and the negative equity boundary.


In [ ]:
def plot_ltv_histogram(df):
    fig, ax = plt.subplots(figsize=(8, 4))

    # clip to realistic LTV range, anything above 105 is dirty data
    for outcome, color in zip(ORDER, COLORS):
        values = df[df[TARGET] == outcome]['loan_to_value_ratio'].dropna()
        values = values[values <= 105]
        ax.hist(values, bins=60, alpha=0.5, color=color,
                label=outcome.replace('_', ' '), density=True)

    ax.axvline(80, color='blue', linestyle='--', linewidth=1, label='LTV = 80%')
    ax.axvline(100, color='black', linestyle='--', linewidth=1, label='LTV = 100%')
    ax.set_title('Loan-to-Value Ratio by Outcome')
    ax.set_xlabel('LTV Ratio (%)')
    ax.set_ylabel('Density')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_ltv_histogram(copy)

### 6.4 Income vs Loan Amount by Outcome

In [ ]:
def plot_scatter_income_loan(df):
    fig, ax = plt.subplots(figsize=(8, 6))

    plot_df = df[(df['income'] >= 0) & (df['income'] <= 500)]
    plot_df = plot_df[plot_df['loan_amount'] <= 1_000_000]

    for outcome, color in zip(ORDER, COLORS):
        subset = plot_df[plot_df[TARGET] == outcome].sample(min(2000, len(plot_df)), random_state=42)
        ax.scatter(subset['income'], subset['loan_amount'],
                   alpha=0.25, s=10, color=color,
                   label=outcome.replace('_', ' '))

    ax.set_xlim(0, 500)
    ax.set_ylim(0, 600_000)
    ax.set_title('Income vs Loan Amount by Outcome')
    ax.set_xlabel('Income ($k)')
    ax.set_ylabel('Loan Amount ($)')
    ax.legend(markerscale=3)
    plt.tight_layout()
    plt.show()

plot_scatter_income_loan(copy)

### 6.5 Loan Type vs Outcome Rate

In [ ]:
def plot_loantype_vs_outcome(df):
    rate_df = df.groupby(['loan_type', TARGET]).size().reset_index(name='count')
    rate_df['rate'] = rate_df['count'] / rate_df.groupby('loan_type')['count'].transform('sum') * 100

    loan_types = df['loan_type'].value_counts().index.tolist()
    x = np.arange(len(loan_types))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))

    for i, (outcome, color) in enumerate(zip(ORDER, COLORS)):
        vals = []
        for lt in loan_types:
            row = rate_df[(rate_df['loan_type'] == lt) & (rate_df[TARGET] == outcome)]['rate']
            vals.append(row.values[0] if len(row) > 0 else 0)
        ax.bar(x + i * width, vals, width, label=outcome.replace('_', ' '), color=color)

    ax.set_xticks(x + width)
    ax.set_xticklabels(loan_types, rotation=15)
    ax.set_title('Loan Type vs Outcome Rate (%)')
    ax.set_ylabel('Percentage (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_loantype_vs_outcome(copy)

### 6.6 Denial Rate by Race

Only racial groups with 100 or more applications are shown to avoid unstable rates from very small samples.  
This chart is included because fairness of lending decisions is a core regulatory concern in HMDA data.


In [ ]:
def plot_denial_by_race(df):
    race_df = df.groupby(['derived_race', TARGET]).size().reset_index(name='count')
    race_df['rate'] = race_df['count'] / race_df.groupby('derived_race')['count'].transform('sum') * 100

    total_per_race = df['derived_race'].value_counts()
    valid_races = total_per_race[total_per_race >= 100].index
    race_df = race_df[race_df['derived_race'].isin(valid_races)]

    denial_rates = (race_df[race_df[TARGET] == 'Application_denied']
                    .set_index('derived_race')['rate']
                    .sort_values(ascending=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(denial_rates.index, denial_rates.values, color='#e74c3c', edgecolor='white')

    for bar, val in zip(bars, denial_rates.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

    ax.set_title('Denial Rate by Race (groups with 100+ applications)')
    ax.set_ylabel('Denial Rate (%)')
    ax.set_xticks(range(len(denial_rates)))
    ax.set_xticklabels(denial_rates.index, rotation=25, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

plot_denial_by_race(copy)

### 6.7 Preapproval Status vs Outcome

In [ ]:
def plot_preapproval_vs_outcome(df):
    pre_df = df.groupby(['preapproval', TARGET]).size().reset_index(name='count')
    pre_df['rate'] = pre_df['count'] / pre_df.groupby('preapproval')['count'].transform('sum') * 100

    preapprovals = df['preapproval'].value_counts().index.tolist()
    x = np.arange(len(preapprovals))
    width = 0.25

    fig, ax = plt.subplots(figsize=(8, 5))

    for i, (outcome, color) in enumerate(zip(ORDER, COLORS)):
        vals = []
        for p in preapprovals:
            row = pre_df[(pre_df['preapproval'] == p) & (pre_df[TARGET] == outcome)]['rate']
            vals.append(row.values[0] if len(row) > 0 else 0)
        ax.bar(x + i * width, vals, width, label=outcome.replace('_', ' '), color=color)

    ax.set_xticks(x + width)
    ax.set_xticklabels(preapprovals)
    ax.set_title('Preapproval Status vs Outcome Rate (%)')
    ax.set_ylabel('Percentage (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_preapproval_vs_outcome(copy)

### 6.8 Neighbourhood Median Income vs Outcome

In [ ]:
def plot_neighborhood_income(df):
    plot_df = df[df['ffiec_msa_md_median_family_income'] > 0]

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=plot_df, x=TARGET, y='ffiec_msa_md_median_family_income',
                order=ORDER, hue=TARGET, palette=COLORS,
                legend=False, ax=ax,
                flierprops=dict(marker='o', markersize=2, alpha=0.3))

    ax.set_title('Neighborhood Median Family Income vs Outcome')
    ax.set_xlabel('')
    ax.set_ylabel('Median Family Income ($)')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Originated', 'Approved Not Accepted', 'Denied'])
    plt.tight_layout()
    plt.show()

plot_neighborhood_income(copy)

### 6.9 Denial Rate vs Tract Minority Population %

This chart bins census tracts by their minority population percentage and shows how denial rates increase with minority concentration — a key fairness signal in HMDA analysis.


In [ ]:
def plot_minority_tract_denial(df):
    temp = df.copy()
    temp['minority_bin'] = pd.cut(temp['tract_minority_population_percent'],
                                   bins=[0, 10, 25, 50, 75, 100],
                                   labels=['0-10%', '10-25%', '25-50%', '50-75%', '75-100%'])

    denial_rates = (temp.groupby('minority_bin', observed=True)
                        .apply(lambda g: (g[TARGET] == 'Application_denied').mean() * 100)
                        .reset_index(name='denial_rate'))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(denial_rates['minority_bin'].astype(str),
            denial_rates['denial_rate'],
            marker='o', color='#e74c3c', linewidth=2)

    ax.set_title('Denial Rate vs Tract Minority Population %')
    ax.set_xlabel('Tract Minority Population')
    ax.set_ylabel('Denial Rate (%)')
    plt.tight_layout()
    plt.show()

plot_minority_tract_denial(copy)

### 6.10 Correlation Matrix

Rate spread and property value are clipped before computing correlations to prevent extreme outliers from distorting the correlation coefficients. Income and loan amount are also clipped to the same ranges used in the boxplots.


In [ ]:
def plot_correlation_heatmap(df):
    num_features = ['loan_amount', 'loan_to_value_ratio', 'interest_rate',
                    'rate_spread', 'total_loan_costs', 'origination_charges',
                    'loan_term', 'property_value', 'income', 'debt_to_income_ratio',
                    'tract_minority_population_percent',
                    'ffiec_msa_md_median_family_income', 'tract_to_msa_income_percentage']

    existing = [c for c in num_features if c in df.columns]

    temp = df[existing].copy()
    temp['rate_spread'] = temp['rate_spread'].clip(-10, 10)
    temp['property_value'] = temp['property_value'].clip(upper=1_000_000)
    temp['income'] = temp['income'].clip(0, 500)
    temp['loan_amount'] = temp['loan_amount'].clip(upper=1_000_000)

    corr = temp.corr()

    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, linewidths=0.4, ax=ax,
                annot_kws={'size': 7})
    ax.set_title('Correlation Matrix - Numeric Features')
    plt.tight_layout()
    plt.show()

plot_correlation_heatmap(copy)

## 7. Target Encoding & Train / Validation / Test Split

`action_taken` is re-encoded numerically for modelling:
- `Loan_originated` → 2 (majority class, ~86%)
- `Approved_but_not_accepted` → 1 (smallest minority)
- `Application_denied` → 0

The data is split into three stratified partitions. Stratification ensures each split preserves the class distribution. The test set (20%) is held out completely and is not touched until the winning model has been selected from validation results.

```
Full data (139,301 rows)
│
├── trainval 80% (111,440 rows)
│   ├── X_train 80% of trainval (~89,152 rows)  ← GridSearchCV fits here
│   └── X_valid 20% of trainval (~22,288 rows)  ← post-tuning model comparison
│
└── X_test 20% (27,861 rows)  ← untouched until winner is chosen
```


In [ ]:

action_map = {
    'Loan_originated':           2,
    'Approved_but_not_accepted': 1,
    'Application_denied':        0
}
copy['action_taken'] = copy['action_taken'].map(action_map)

X = copy.drop(columns=['action_taken'])
y = copy['action_taken']

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y_trainval)

print(f"X_train : {X_train.shape[0]} rows")
print(f"X_valid : {X_valid.shape[0]} rows")
print(f"X_test  : {X_test.shape[0]} rows")
print(f"\nClass distribution in y_train:\n{y_train.value_counts(normalize=True).round(3)}")

## 8. Modelling

All modelling functions are imported from `modeler.py`. The file contains:
- `preprocessor_scaled` and `preprocessor_not_scaled` — sklearn Pipelines that apply imputation and optional scaling. Both are fitted only on training data via GridSearchCV, preventing leakage.
- `baseline_model()` — DummyClassifier (most_frequent strategy)
- `tune_log_reg()`, `tune_dec_tree()`, `tune_r_forest()`, `tune_xgboost()` — each runs GridSearchCV on training data, evaluates the best estimator on validation, and plots feature importances
- `plot_confusion_matrix()` — plots a confusion matrix for any fitted model
- `evaluate_winner_on_test()` — finds the best threshold on validation, then evaluates the winner on the test set exactly once

**Metric choice:** Macro F1 is used as the primary selection metric. Accuracy is dominated by the majority class (~86%) and is misleading for model selection. Macro F1 weights all three classes equally, penalising models that ignore minority classes — which is both statistically correct and aligned with the regulatory intent of HMDA data.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, classification_report)

RANDOM_STATE = 42

numeric_cols = [
    'loan_amount', 'loan_to_value_ratio', 'loan_term', 'property_value',
    'income', 'debt_to_income_ratio', 'tract_population',
    'tract_minority_population_percent', 'ffiec_msa_md_median_family_income',
    'tract_to_msa_income_percentage', 'tract_owner_occupied_units',
    'tract_one_to_four_family_homes', 'tract_median_age_of_housing_units',
    'applicant_age_not_applicable', 'applicant_age_ord',
    'coapplicant_age_not_applicable', 'coapplicant_age_ord',
    'is_single_family', 'no_coapplicant'
]

categorical_cols = [
    'conforming_loan_limit', 'derived_loan_product_type', 'derived_ethnicity',
    'derived_race', 'derived_sex', 'preapproval', 'loan_type', 'lien_status',
    'reverse_mortgage', 'open-end_line_of_credit', 'business_or_commercial_purpose',
    'negative_amortization', 'interest_only_payment', 'balloon_payment',
    'other_nonamortizing_features', 'construction_method', 'occupancy_type',
    'manufactured_home_secured_property_type', 'manufactured_home_land_property_interest',
    'applicant_credit_score_type', 'co-applicant_credit_score_type',
    'applicant_ethnicity-1', 'co-applicant_ethnicity-1', 'applicant_ethnicity_observed',
    'co-applicant_ethnicity_observed', 'applicant_race-1', 'co-applicant_race-1',
    'applicant_race_observed', 'co-applicant_race_observed', 'applicant_sex',
    'co-applicant_sex', 'applicant_sex_observed', 'co-applicant_sex_observed',
    'applicant_age_above_62', 'submission_of_application',
    'initially_payable_to_institution', 'aus-1', 'county_code',
    'derived_dwelling_category', 'derived_msa-md'
]

## Preprocessing Pipelines

In [ ]:
numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

numeric_not_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor_scaled = ColumnTransformer(transformers=[
    ('num', numeric_scaled, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)], remainder='drop')

preprocessor_not_scaled = ColumnTransformer(transformers=[
    ('num', numeric_not_scaled, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)], remainder='drop')

## Evaluation Function

In [ ]:
def evaluator(name, model, X_true, y_true):
    pred = model.predict(X_true)
    probabilities = model.predict_proba(X_true)

    acc = accuracy_score(y_true, pred)
    bal_acc = balanced_accuracy_score(y_true, pred)
    prec = precision_score(y_true, pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, pred, average='weighted', zero_division=0)
    macro_f1 = f1_score(y_true, pred, average='macro', zero_division=0)
    roc_auc = roc_auc_score(y_true, probabilities, multi_class='ovr', average='macro')

    report_str = classification_report(y_true, pred, zero_division=0)


    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), gridspec_kw={'width_ratios': [1, 1.2]})
    fig.suptitle(f'Model Evaluation Report: {name}', fontsize=14, fontweight='bold', y=1.02)

    metrics_dict = {
        'Accuracy': acc,
        'Balanced Acc': bal_acc,
        'Weighted F1': f1,
        'Macro F1': macro_f1,
        'ROC-AUC (Macro)': roc_auc
    }
    metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])

    sns.barplot(data=metrics_df, x='Score', y='Metric', hue='Metric', legend=False,
                ax=axes[0], palette='Blues_r')
    axes[0].set_xlim(0, 1.0)
    axes[0].set_title('Core Performance Metrics', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Score Value')
    axes[0].set_ylabel('')

    for p in axes[0].patches:
        width = p.get_width()
        axes[0].text(width + 0.02, p.get_y() + p.get_height() / 2, f'{width:.3f}',
                     va='center', ha='left', fontsize=10, fontweight='bold')

    report_dict = classification_report(y_true, pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).iloc[:-1, :-3].T

    sns.heatmap(report_df, annot=True, cmap='Blues', fmt='.2f', cbar=False,
                ax=axes[1], linewidths=0.5)
    axes[1].set_title('Class-wise Performance (Precision, Recall, F1)',
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Metrics')
    axes[1].set_ylabel('Classes')

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    return {
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "macro_f1": macro_f1,
        "roc_auc": roc_auc,
        "classification_report": report_str}

## Confusion Matrix

In [ ]:
def plot_confusion_matrix(model, X_valid, y_valid, model_name):
    y_pred = model.predict(X_valid)
    cm = confusion_matrix(y_valid, y_pred, labels=[0, 1, 2])

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1, 2])
    disp.plot(cmap='Blues', values_format='d', ax=ax, colorbar=True)
    ax.set_title(f'Confusion Matrix - {model_name}', fontsize=14, pad=15)
    fig.tight_layout()
    plt.show()

    return cm

## Feature Importance

In [ ]:
def plot_feature_importance(model, model_name, top_n=20):
    preprocessor = model.named_steps['preprocessor']
    classifier = model.named_steps['model']

    try:
        ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
        cat_feature_names = ohe.get_feature_names_out(categorical_cols).tolist()
    except Exception:
        cat_feature_names = []

    feature_names = numeric_cols + cat_feature_names

    # Decision Tree, Random Forest, XGBoost
    if hasattr(classifier, 'feature_importances_'):
        importances = classifier.feature_importances_
        importance_label = 'Feature Importance (Gini / Gain)'

        # Logistic Regression - multiclass gives one row, takes the mean abs
    elif hasattr(classifier, 'coef_'):
        importances = np.abs(classifier.coef_).mean(axis=0)
        importance_label = 'Mean |Coefficient| Across Classes'
    else:
        fig, ax = plt.subplots(figsize=(6, 0.4))
        ax.axis('off')
        ax.text(0.5, 0.5, f'Feature importance not available for {type(classifier).__name__}',
                ha='center', va='center', fontsize=12, color='grey',
                transform=ax.transAxes)
        plt.tight_layout()
        plt.show()
        return None

    importance_df = pd.DataFrame({
        'Feature': feature_names[:len(importances)],
        'Importance': importances
    }).sort_values('Importance', ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Importance', y='Feature',
                hue='Feature', legend=False, palette='Blues_r', ax=ax)
    ax.set_title(f'Top {top_n} Feature Importances — {model_name}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(importance_label)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
    return importance_df


## Threshold Analysis

In [ ]:
def find_best_threshold(model, X_valid, y_valid, model_name, target_class):
    thresholds = np.arange(0.10, 0.90, 0.05)
    probabilities = model.predict_proba(X_valid)

    results = []
    for thresh in thresholds:
        preds = model.predict(X_valid).copy()
        preds[probabilities[:, target_class] >= thresh] = target_class

        f1_macro = f1_score(y_valid, preds, average='macro', zero_division=0)
        recall_target = recall_score(y_valid, preds, average=None,
                                     zero_division=0)[target_class]
        results.append({
            'threshold': round(thresh, 2),
            'macro_f1': f1_macro,
            f'recall_class_{target_class}': recall_target
        })

    results_df = pd.DataFrame(results)
    best_row = results_df.loc[results_df['macro_f1'].idxmax()]
    best_threshold = best_row['threshold']

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(results_df['threshold'], results_df['macro_f1'],
            marker='o', color='steelblue', label='Macro F1')
    ax.plot(results_df['threshold'], results_df[f'recall_class_{target_class}'],
            marker='s', color='salmon', linestyle='--',
            label=f'Recall Class {target_class}')
    ax.axvline(best_threshold, color='green', linestyle=':', linewidth=2,
               label=f'Best threshold = {best_threshold:.2f}')
    ax.set_title(f'Threshold Analysis — {model_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.legend()
    plt.tight_layout()
    plt.show()

    summary_df = pd.DataFrame([{
        'Model': model_name,
        'Target Class': target_class,
        'Best Threshold': f"{best_threshold:.2f}",
        'Macro F1 at Best Threshold': f"{best_row['macro_f1']:.4f}",
        f'Recall Class {target_class} at Best Threshold': f"{best_row[f'recall_class_{target_class}']:.4f}"}])

    fig, ax = plt.subplots(figsize=(10, 1.2))
    ax.axis('off')
    tbl = ax.table(
        cellText=summary_df.values,
        colLabels=summary_df.columns,
        cellLoc='center',
        loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c5f8a')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#d9eaf7')
    ax.set_title('Threshold Selection Summary', fontsize=11,
                 fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

    return best_threshold, results_df


def predict_with_threshold(model, X, threshold, target_class):
    probabilities = model.predict_proba(X)
    preds = model.predict(X).copy()
    preds[probabilities[:, target_class] >= threshold] = target_class
    return preds

### 8.1 Baseline Model

A `DummyClassifier(strategy='most_frequent')` predicts the majority class (Loan Originated) for every observation. With ~86% class 2, it achieves ~86% accuracy by never predicting a minority class. Any real model must beat this in macro F1 to demonstrate genuine learning.


In [ ]:
def baseline_model(X_train, y_train, X_valid, y_valid):
    dummy = Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', DummyClassifier(strategy="most_frequent"))])

    dummy.fit(X_train, y_train)
    baseline = evaluator('Dummy Classifier', dummy, X_valid, y_valid)
    return baseline


baseline = baseline_model(X_train, y_train, X_valid, y_valid)


print("--- Model Metrics ---")
for key, value in baseline.items():
    if key != 'classification_report':
        print(f"{key}: {value}")


print("\n--- Classification Report ---")
print(baseline['classification_report'])

### 8.2 Logistic Regression (Tuned)

Logistic Regression is included as the simplest interpretable classifier above the dummy baseline. It is appropriate for large tabular datasets and provides coefficient-based interpretability.

**Known limitation:** Logistic Regression relies on a linear decision boundary. With severe class imbalance on complex, non-linear data (financial interactions between income, DTI, loan type, race), it hits a hard performance ceiling on minority classes. `class_weight='balanced'` was tested and found to cause a surge of false positives for this linear model — a finding that justifies the inclusion of tree-based models.

**Grid:** C ∈ {0.01, 1, 100} × class_weight ∈ {None, balanced} = 6 combos × 3 CV = 18 fits


In [ ]:
def tune_log_reg(X_train, y_train, X_valid, y_valid):
    log_reg_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE))])

    params = {
        'model__C': [0.01, 1, 10, 100],
        'model__class_weight': [None, 'balanced']
    }

    grid_search = GridSearchCV(
        estimator=log_reg_pipeline,
        param_grid=params,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1)

    grid_search.fit(X_train, y_train)
    results_df = pd.DataFrame(grid_search.cv_results_)

    eval_results = evaluator('Logistic Regression (Tuned)',
                             grid_search.best_estimator_, X_valid, y_valid)

    plot_feature_importance(grid_search.best_estimator_,
                            'Logistic Regression (Tuned)')
    return grid_search.best_estimator_, grid_search.best_params_, results_df, eval_results


print("\n")
log_model, log_params, log_cv, log_eval = tune_log_reg(X_train, y_train, X_valid, y_valid)
print("\n")
plot_confusion_matrix(log_model, X_valid, y_valid, 'Logistic Regression (Tuned)')
print("\n")
print(log_params)

### 8.3 Decision Tree (Tuned)

A single Decision Tree is included to illustrate how the base learner behind ensemble methods behaves in isolation. It is interpretable and fast, but prone to overfitting and instability.

**Known limitation:** When `class_weight='balanced'` is applied, the tree alters split thresholds aggressively to capture rare cases, introducing a large wave of false positives and degrading overall accuracy. This contrast with Random Forest demonstrates why ensembling is necessary.

**Grid:** max_depth ∈ {5, 10, 20} × min_samples_leaf ∈ {1, 5} × criterion ∈ {gini, entropy} × class_weight ∈ {balanced} = 12 combos × 3 CV = 36 fits


In [ ]:
def tune_dec_tree(X_train, y_train, X_valid, y_valid):
    dec_tree_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_not_scaled),
        ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))])

    params = {
        'model__max_depth': [5, 10, 20],
        'model__class_weight': [None, 'balanced'],
        'model__min_samples_leaf': [1, 5],
        'model__criterion': ['gini', 'entropy']}

    grid_search = GridSearchCV(
        estimator=dec_tree_pipeline,
        param_grid=params,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1)

    grid_search.fit(X_train, y_train)
    dt_results_df = pd.DataFrame(grid_search.cv_results_)

    eval_results = evaluator('Decision Tree (Tuned)',
                             grid_search.best_estimator_, X_valid, y_valid)

    plot_feature_importance(grid_search.best_estimator_,
                            'Decision Tree (Tuned)')

    return grid_search.best_estimator_, grid_search.best_params_, dt_results_df, eval_results

print("\n")
dt_model, dt_params, dt_cv, dt_eval = tune_dec_tree(X_train, y_train, X_valid, y_valid)
print("\n")
plot_confusion_matrix(dt_model, X_valid, y_valid, 'Decision Tree (Tuned)')
print("\n")
print(dt_params)

### 8.4 Random Forest (Tuned)

Random Forest is the primary ensemble candidate. It handles mixed feature types, is robust to outliers, and can capture non-linear interactions between features (e.g. income × loan type × race) that Logistic Regression cannot.

`class_weight='balanced'` improved Random Forest performance on minority classes where it did not help Logistic Regression — tree-based models create branching, non-linear rules that can isolate minority patterns without generating a massive wave of false positives.

`class_weight='balanced_subsample'` recomputes weights per bootstrap sample and was also searched.

**Grid:** n_estimators ∈ {100, 300} × max_depth ∈ {10, None} × class_weight ∈ {balanced, balanced_subsample} × max_features ∈ {sqrt} = 8 combos × 3 CV = 24 fits


In [ ]:
def tune_r_forest(X_train, y_train, X_valid, y_valid):
    r_forest_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_not_scaled),
        ('model', RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1))])

    params = {
        'model__n_estimators': [200, 300],
        'model__max_depth': [10, 20, None],
        'model__class_weight': ['balanced', 'balanced_subsample'],
        'model__max_features': ['sqrt'],
        'model__min_samples_leaf': [1, 2]
    }

    grid_search = GridSearchCV(
        estimator=r_forest_pipeline,
        param_grid=params,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1)

    grid_search.fit(X_train, y_train)
    r_forest_results_df = pd.DataFrame(grid_search.cv_results_)

    eval_results = evaluator('Random Forest (Tuned)',
                             grid_search.best_estimator_, X_valid, y_valid)

    plot_feature_importance(grid_search.best_estimator_,
                            'Random Forest (Tuned)')

    return grid_search.best_estimator_, grid_search.best_params_, r_forest_results_df, eval_results

print("\n")
rf_model, rf_params, rf_cv, rf_eval = tune_r_forest(X_train, y_train, X_valid, y_valid)
print("\n")
plot_confusion_matrix(rf_model, X_valid, y_valid, 'Random Forest (Tuned)')
print("\n")
print(rf_params)

### 8.5 XGBoost (Tuned)

XGBoost is the strongest candidate. It sequentially corrects errors from previous trees, handles structured tabular data well, and typically outperforms Random Forest by focusing on the hardest-to-classify examples.

**Grid:** n_estimators ∈ {100, 300} × max_depth ∈ {3, 5} × learning_rate ∈ {0.05, 0.1, 0.2} = 12 combos × 3 CV = 36 fits


In [ ]:
def tune_xgboost(X_train, y_train, X_valid, y_valid):
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    xgboost_pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor_not_scaled),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            max_delta_step=1))
    ])
    params = {
        'model__n_estimators': [300, 500],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.05, 0.1],
        'model__min_child_weight': [1, 5, 10],
        'model__subsample': [0.8]
    }
    grid_search = GridSearchCV(
        estimator=xgboost_pipeline,
        param_grid=params,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1)

    grid_search.fit(X_train, y_train)
    xgboost_results_df = pd.DataFrame(grid_search.cv_results_)

    eval_results = evaluator('XGBoost (Tuned)',
                             grid_search.best_estimator_, X_valid, y_valid)

    plot_feature_importance(grid_search.best_estimator_,
                            'XGBoost (Tuned)')

    return grid_search.best_estimator_, grid_search.best_params_, xgboost_results_df, eval_results

print("\n")
xgb_model, xgb_params, xgb_cv, xgb_eval = tune_xgboost(X_train, y_train, X_valid, y_valid)
print("\n")
plot_confusion_matrix(xgb_model, X_valid, y_valid, 'XGBoost (Tuned)')
print("\n")
print(xgb_params)

## 9. Model Comparison on Validation Set

All four tuned models are compared on the same validation set using the same metrics. The winner is selected based on macro F1 — the primary metric. The test set is not consulted at this stage.


In [ ]:
comparison = pd.DataFrame([log_eval, dt_eval, rf_eval, xgb_eval])
(comparison[['model', 'macro_f1', 'roc_auc', 'balanced_accuracy']]
 .sort_values(by='macro_f1', ascending=False).reset_index(drop=True))

## 10. Final Test Evaluation

The winning model (XGBoost) is evaluated on the test set **exactly once**.

Before touching the test set, the optimal classification threshold for the minority class (class 2 - Loan_originated) is found using the validation set. This threshold is then applied to the test set predictions. The test set is never used to select the threshold.


In [ ]:
def evaluate_winner_on_test(winner_model, winner_name, X_valid, y_valid,
                             X_test, y_test, target_class):

    # Find best threshold on validation
    best_threshold, threshold_df = find_best_threshold(
        winner_model, X_valid, y_valid, winner_name, target_class=target_class)

    # Apply threshold to test set - TEST SIZE
    y_pred_test = predict_with_threshold(
        winner_model, X_test, best_threshold, target_class=target_class)

    # Confusion matrix on test
    cm = confusion_matrix(y_test, y_pred_test, labels=[0, 1, 2])
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1, 2])
    disp.plot(cmap='Blues', values_format='d', ax=ax, colorbar=True)
    ax.set_title(f'Test Set Confusion Matrix — {winner_name}', fontsize=14, pad=15)
    fig.tight_layout()
    plt.show()

    # Final metrics on test
    test_macro_f1 = f1_score(y_test, y_pred_test, average='macro', zero_division=0)
    test_bal_acc  = balanced_accuracy_score(y_test, y_pred_test)
    test_weighted_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    test_roc_auc  = roc_auc_score(
        y_test, winner_model.predict_proba(X_test), multi_class='ovr', average='macro')

    metrics_dict = {
        'Balanced Accuracy': test_bal_acc,
        'Weighted F1':       test_weighted_f1,
        'Macro F1':          test_macro_f1,
        'ROC-AUC (Macro)':   test_roc_auc}

    metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.4]})
    fig.suptitle(f'Final Test Results — {winner_name}  |  Threshold {best_threshold:.2f} '
                 f'on class {target_class}', fontsize=14, fontweight='bold', y=1.01)

    sns.barplot(data=metrics_df, x='Score', y='Metric', hue='Metric',
                legend=False, ax=axes[0], palette='Blues_r')
    axes[0].set_xlim(0, 1.0)
    axes[0].set_title('Headline Metrics (Test Set)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Score')
    axes[0].set_ylabel('')
    for p in axes[0].patches:
        w = p.get_width()
        axes[0].text(w + 0.02, p.get_y() + p.get_height() / 2,
                     f'{w:.4f}', va='center', ha='left', fontsize=10, fontweight='bold')

    report_dict = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).iloc[:-1, :-3].T
    sns.heatmap(report_df, annot=True, cmap='Blues', fmt='.2f', cbar=False,
                ax=axes[1], linewidths=0.5)
    axes[1].set_title('Per-Class Report (Test Set)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Metrics')
    axes[1].set_ylabel('Classes')

    plt.tight_layout()
    plt.show()

    # Summary table
    summary_data = {
        'Model': [winner_name],
        'Set': ['Test'],
        'Threshold': [f"{best_threshold:.2f}"],
        'Macro F1': [f"{test_macro_f1:.4f}"],
        'Balanced Acc': [f"{test_bal_acc:.4f}"],
        'Weighted F1': [f"{test_weighted_f1:.4f}"],
        'ROC-AUC': [f"{test_roc_auc:.4f}"]
    }
    summary_df = pd.DataFrame(summary_data)

    fig2, ax2 = plt.subplots(figsize=(12, 1.2))
    ax2.axis('off')
    tbl = ax2.table(
        cellText=summary_df.values,
        colLabels=summary_df.columns,
        cellLoc='center',
        loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.8)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#1a3f5c')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#cce0f0')
    ax2.set_title('Final Test Score Summary', fontsize=11, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

    summary_df = pd.DataFrame(summary_data)
    print("\n--- Final Test Score Summary ---")
    print(summary_df.to_string(index=False))
    print("-" * 50)

    return {
        'best_threshold': best_threshold,
        'threshold_analysis': threshold_df,
        'confusion_matrix': cm,
        'macro_f1': test_macro_f1,
        'roc_auc': test_roc_auc}

test_results = evaluate_winner_on_test(
    winner_model=rf_model,
    winner_name='Random Forest',
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    target_class=2
)

In [ ]:
params_summary = pd.DataFrame([
    {'Model': 'Logistic Regression', **log_params},
    {'Model': 'Decision Tree',       **dt_params},
    {'Model': 'Random Forest',       **rf_params},
    {'Model': 'XGBoost',             **xgb_params},
])

params_summary